In [13]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

# ==========================================
# 1. CONFIGURATION & DATA PREPARATION
# ==========================================

# File path variable for your MNAR dataset
DATASET_PATH = r"C:\Users\night\Downloads\Datasets\banknote\data_banknote_authentication.txt"
TARGET_COLUMN = "class" # Change this to your actual target column name

# Hyperparameters
BATCH_SIZE = 64
EPOCHS = 100
LEARNING_RATE = 1e-3
HIDDEN_DIM = 64 # Size of the hidden layers in our neural networks


#Assuming BATCH_SIZE is globally defined as in the previous code
BATCH_SIZE = 64

def load_and_prepare_data(filepath, target_col, mechanism=None, missing_rate=0.5):
    """
    Loads data and simulates structured MNAR missingness based on the chosen mechanism.
    
    Valid mechanisms: 'self_masking', 'logistic', 'sequential_logistic', or None
    """
    df = pd.read_csv(filepath)
    print(df.head())
    # Separate features (X) and target (y)
    y = df[target_col].values
    X = df.drop(columns=[target_col]).values
    
    N, d = X.shape
    
    # Create the binary mask (1 for observed, 0 for missing)
    mask = np.ones((N, d))
    
    # Helper to calculate sigmoid avoiding overflow
    def sigmoid(z):
        return 1 / (1 + np.exp(-np.clip(z, -10, 10)))
        
    # Standardize data locally just for generating stable missingness probabilities 
    X_std = (X - np.mean(X, axis=0)) / (np.std(X, axis=0) + 1e-8)
    y_std = (y - np.mean(y)) / (np.std(y) + 1e-8)
    
    # Apply selected MNAR simulation technique using if/elif blocks
    if mechanism == 'self_masking':
        # 1. Self-Masking Mechanism
        # Probability of missingness depends only on the value of that covariate itself
        for j in range(d):
            psi_1 = np.random.randn()
            term = psi_1 * X_std[:, j]
            
            # Adjust psi_0 (bias) to hit the approximate target missing rate
            psi_0 = np.log(missing_rate / (1 - missing_rate)) - np.mean(term)
            
            prob_missing = sigmoid(psi_0 + term)
            r_ij = np.random.binomial(1, prob_missing)
            mask[:, j] = 1.0 - r_ij

    elif mechanism == 'logistic':
        # 2. Logistic Mechanism
        # Missingness depends jointly on all covariates and the response
        for j in range(d):
            psi_k = np.random.randn(d)
            psi_y = np.random.randn()
            
            term = np.dot(X_std, psi_k) + psi_y * y_std
            
            psi_0 = np.log(missing_rate / (1 - missing_rate)) - np.mean(term)
            prob_missing = sigmoid(psi_0 + term)
            
            r_ij = np.random.binomial(1, prob_missing)
            mask[:, j] = 1.0 - r_ij

    elif mechanism == 'sequential_logistic':
        # 3. Sequential Logistic (ReverseChain) Mechanism
        # Captures inter-feature dependencies, generated sequentially in reverse order
        R_matrix = np.zeros((N, d)) # Temporarily stores the missingness indicators (r_ik)
        
        # Python indices iterate from d-1 down to 0
        for j in range(d - 1, -1, -1):
            psi_x = np.random.randn(d)
            psi_y = np.random.randn()
            
            # Base equation includes y and all x features
            eta_ij_base = np.dot(X_std, psi_x) + psi_y * y_std
            
            # Add dependency on previously generated missingness indicators (r_k for k > j)
            if j < d - 1:
                psi_r = np.random.randn(d - 1 - j)
                eta_ij_base += np.dot(R_matrix[:, j+1:], psi_r)
                
            psi_0 = np.log(missing_rate / (1 - missing_rate)) - np.mean(eta_ij_base)
            prob_missing = sigmoid(psi_0 + eta_ij_base)
            
            R_matrix[:, j] = np.random.binomial(1, prob_missing)
            mask[:, j] = 1.0 - R_matrix[:, j]
            
    elif mechanism is not None:
        raise ValueError("Invalid mechanism. Choose 'self_masking', 'logistic', or 'sequential_logistic'.")

    # Apply the generated mask: Temporarily fill missing slots with 0.0
    X_filled = X * mask 
    
    # 80-20 Train-Test Split
    X_train, X_test, y_train, y_test, m_train, m_test = train_test_split(
        X_filled, y, mask, test_size=0.20, random_state=42
    )
    
    # Convert to PyTorch Tensors
    train_dataset = TensorDataset(
        torch.FloatTensor(X_train), 
        torch.FloatTensor(y_train), 
        torch.FloatTensor(m_train)
    )
    test_dataset = TensorDataset(
        torch.FloatTensor(X_test), 
        torch.FloatTensor(y_test), 
        torch.FloatTensor(m_test)
    )
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    return train_loader, test_loader, d

#Example usage when setting up the model later:
train_loader, test_loader, input_dim = load_and_prepare_data(
    filepath = DATASET_PATH, 
    target_col = TARGET_COLUMN, 
    mechanism = "self_masking", # Pass your chosen mechanism here
    missing_rate = 0.2
)

   variance  skewness  curtosis  entropy  class
0   3.62160    8.6661   -2.8073 -0.44699      0
1   4.54590    8.1674   -2.4586 -1.46210      0
2   3.86600   -2.6383    1.9242  0.10645      0
3   3.45660    9.5228   -4.0112 -3.59440      0
4   0.32924   -4.4552    4.5718 -0.98880      0


In [8]:
# def load_and_prepare_data(filepath, target_col):
#     """
#     Loads data, isolates the target, creates missingness masks, and splits 80/20.
#     """
#     df = pd.read_csv(filepath,delimiter=";")
#     print(df)
#     # Separate features (X) and target (y)
#     y = df[target_col].values
#     X_df = df.drop(columns=[target_col])
    
#     # Create the binary mask (1 for observed, 0 for missing)
#     # We need this to tell the network which values to sample and which to keep.
#     mask = (~X_df.isna()).astype(float).values
    
#     # Temporarily fill NaNs with 0 so PyTorch can process the tensors.
#     # The network won't be fooled because the mask tells it these 0s are fake.
#     X_filled = X_df.fillna(0.0).values
    
#     # 80-20 Train-Test Split
#     X_train, X_test, y_train, y_test, m_train, m_test = train_test_split(
#         X_filled, y, mask, test_size=0.20, random_state=42
#     )
    
#     # Convert to PyTorch Tensors
#     train_dataset = TensorDataset(
#         torch.FloatTensor(X_train), 
#         torch.FloatTensor(y_train), 
#         torch.FloatTensor(m_train)
#     )
#     test_dataset = TensorDataset(
#         torch.FloatTensor(X_test), 
#         torch.FloatTensor(y_test), 
#         torch.FloatTensor(m_test)
#     )
    
#     train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
#     test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
#     return train_loader, test_loader, X_train.shape[1]

In [ ]:
# ==========================================
# 2. ARCHITECTURE DEFINITION
# ==========================================

class DLGLM(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(DLGLM, self).__init__()
        
        # --- THE ENCODER (Secondary Network for Missing Data) ---
        # Takes both the zero-filled data AND the mask as input so it knows 
        # exactly which features are missing. Outputting mu and log(variance).
        self.encoder = nn.Sequential(
            nn.Linear(input_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )
        # We predict log(variance) instead of standard variance because 
        # neural networks can output negative numbers, but variance must be positive.
        # Taking the exponent of log(variance) guarantees a positive number.
        self.enc_mu = nn.Linear(hidden_dim, input_dim)
        self.enc_logvar = nn.Linear(hidden_dim, input_dim)
        
        # --- THE MAIN NN (Deep Linear Predictor) ---
        # This replaces the standard linear component of a classical GLM.
        # It maps the completed dataset to the final prediction.
        self.decoder_glm = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            # Output layer: 1 node for Regression (Gaussian GLM). 
            # If doing Classification (Binomial GLM), add nn.Sigmoid() here.
            nn.Linear(hidden_dim, 1) 
        )

    def reparameterize(self, mu, logvar):
        """
        The Reparameterization Trick: z = mu + sigma * epsilon
        Allows gradients to backpropagate through the random sampling step.
        """
        std = torch.exp(0.5 * logvar) # Convert log(variance) back to standard deviation
        epsilon = torch.randn_like(std) # Sample random noise from standard Normal N(0,1)
        return mu + epsilon * std

    def forward(self, x, mask):
        # 1. Concatenate input with mask to feed the Encoder
        enc_input = torch.cat([x, mask], dim=1)
        
        # 2. Estimate the probability distribution parameters for every feature
        hidden_enc = self.encoder(enc_input)
        mu = self.enc_mu(hidden_enc)
        logvar = self.enc_logvar(hidden_enc)
        
        # 3. Sample customized values from the learned distributions
        z_sampled = self.reparameterize(mu, logvar)
        
        # 4. Construct the Complete Dataset
        # If mask == 1 (observed), keep the real data 'x'.
        # If mask == 0 (missing), insert the sampled data 'z_sampled'.
        x_complete = (x * mask) + (z_sampled * (1 - mask))
        
        # 5. Main GLM Prediction
        y_pred = self.decoder_glm(x_complete)
        
        return y_pred, mu, logvar

# ==========================================
# 3. THE LOSS FUNCTION (ELBO)
# ==========================================

def elbo_loss(y_pred, y_true, mu, logvar, mask):
    """
    Evidence Lower Bound (ELBO) = Prediction Loss + KL Divergence.
    """
    # 1. Prediction Loss (Negative Log-Likelihood)
    # For a Gaussian GLM (Regression), this is mathematically equivalent to Mean Squared Error.
    # For a Binomial GLM (Classification), you would use Binary Cross Entropy (BCE).
    mse_loss = nn.MSELoss()(y_pred.squeeze(), y_true)
    
    # 2. KL Divergence
    # Measures how far the learned distributions are from the standard Normal prior N(0,1).
    # Formula: -0.5 * sum(1 + log(sigma^2) - mu^2 - sigma^2)
    kl_elements = -0.5 * (1 + logvar - mu.pow(2) - logvar.exp())
    
    # CRITICAL STEP: We ONLY calculate KL divergence for the missing values.
    # We multiply by (1 - mask) to zero out the KL penalty for observed features.
    kl_masked = kl_elements * (1 - mask)
    
    # Sum across the feature dimension, then take the mean across the batch
    kl_loss = torch.sum(kl_masked, dim=1).mean()
    
    # Total Loss
    total_loss = mse_loss + kl_loss
    return total_loss, mse_loss, kl_loss

In [14]:
# ==========================================
# 4. TRAINING LOOP
# ==========================================

def train_model(train_loader, input_dim):
    # Load Data
    # Note: Uncomment the next line and provide the dataset before running.
    # train_loader, test_loader, input_dim = load_and_prepare_data(DATASET_PATH, TARGET_COLUMN)
    
    
    # Initialize Model and Optimizer
    model = DLGLM(input_dim=input_dim, hidden_dim=HIDDEN_DIM)
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    
    print("Starting Training...")
    for epoch in range(EPOCHS):
        model.train()
        epoch_loss = 0
        epoch_mse = 0
        epoch_kl = 0
        
        for batch_idx, (x_batch, y_batch, mask_batch) in enumerate(train_loader):
            optimizer.zero_grad()
            
            # Forward Pass
            y_pred, mu, logvar = model(x_batch, mask_batch)
            
            # Compute ELBO
            loss, mse, kl = elbo_loss(y_pred, y_batch, mu, logvar, mask_batch)
            
            # Backward Pass (Gradient flow through both networks via reparameterization)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            epoch_mse += mse.item()
            epoch_kl += kl.item()
            
        # Print progress
        if (epoch + 1) % 10 == 0:
            print(f"Epoch [{epoch+1}/{EPOCHS}] | Total Loss: {epoch_loss/len(train_loader):.4f} | "
                  f"MSE: {epoch_mse/len(train_loader):.4f} | KL: {epoch_kl/len(train_loader):.4f}")

    print("Training Complete.")
    return model

# To run the code after filling the dataset path:
trained_model = train_model(train_loader,input_dim)

Starting Training...
Epoch [10/100] | Total Loss: 0.0888 | MSE: 0.0879 | KL: 0.0009
Epoch [20/100] | Total Loss: 0.0807 | MSE: 0.0797 | KL: 0.0011
Epoch [30/100] | Total Loss: 0.0781 | MSE: 0.0767 | KL: 0.0014
Epoch [40/100] | Total Loss: 0.0764 | MSE: 0.0749 | KL: 0.0015
Epoch [50/100] | Total Loss: 0.0788 | MSE: 0.0772 | KL: 0.0016
Epoch [60/100] | Total Loss: 0.0678 | MSE: 0.0662 | KL: 0.0016
Epoch [70/100] | Total Loss: 0.0707 | MSE: 0.0695 | KL: 0.0012
Epoch [80/100] | Total Loss: 0.0774 | MSE: 0.0761 | KL: 0.0013
Epoch [90/100] | Total Loss: 0.0683 | MSE: 0.0668 | KL: 0.0015
Epoch [100/100] | Total Loss: 0.0618 | MSE: 0.0604 | KL: 0.0014
Training Complete.


In [17]:
import numpy as np
import torch
from sklearn.metrics import roc_auc_score

def predict_proba_mc(model, dataloader, num_mc_samples=25, device="cpu"):
    """
    Computes expected probabilities by Monte Carlo averaging over S posterior 
    samples for missing values: E_q(z)[p(y|z, x_obs)].
    """
    model.eval()
    model.to(device)
    
    all_y_true = []
    all_y_probs = []
    
    with torch.no_grad():
        for x_batch, y_batch, mask_batch in dataloader:
            x_batch = x_batch.to(device)
            mask_batch = mask_batch.to(device)
            
            # Accumulator for Monte Carlo predicted probabilities
            mc_batch_probs = torch.zeros(x_batch.size(0), 1, device=device)
            
            for _ in range(num_mc_samples):
                # 1. Forward pass (generates dynamic z_sampled via reparameterization)
                y_pred_logits, _, _ = model(x_batch, mask_batch)
                
                # 2. Map logits to probabilities via Sigmoid (for Binary GLM)
                # If your model decoder already has nn.Sigmoid(), omit torch.sigmoid()
                prob = torch.sigmoid(y_pred_logits)
                mc_batch_probs += prob
                
            # Average across the S Monte Carlo samples
            avg_batch_prob = mc_batch_probs / num_mc_samples
            
            all_y_probs.append(avg_batch_prob.cpu().numpy().squeeze())
            all_y_true.append(y_batch.numpy().squeeze())
            
    y_true = np.concatenate(all_y_true)
    y_probs = np.concatenate(all_y_probs)
    return y_true, y_probs


def evaluate_auc_distribution(model, test_loader, n_trials=10, num_mc_samples=25, device="cpu"):
    """
    Evaluates ROC-AUC across repeated Monte Carlo evaluations / bootstrap runs
    to compute Mean ± Standard Deviation.
    """
    auc_scores = []
    
    print(f"Running {n_trials} evaluation trials")
    
    for trial in range(n_trials):
        y_true, y_probs = predict_proba_mc(
            model=model, 
            dataloader=test_loader, 
            num_mc_samples=num_mc_samples, 
            device=device
        )
        
        # Calculate ROC-AUC for this trial
        trial_auc = roc_auc_score(y_true, y_probs)
        auc_scores.append(trial_auc)
        print(f"  Trial {trial + 1}/{n_trials} AUC: {trial_auc:.4f}")
        
    auc_scores = np.array(auc_scores)
    mean_auc = np.mean(auc_scores)
    std_auc = np.std(auc_scores)
    
    print("\n" + "=" * 45)
    print(f"Final Model Performance (ROC-AUC): {mean_auc:.4f} ± {std_auc:.4f}")
    print("=" * 45)
    
    return mean_auc, std_auc, auc_scores



mean_auc, std_auc, all_scores = evaluate_auc_distribution(
    model=trained_model,
    test_loader=test_loader,
    n_trials=10,          # 10 independent evaluation trials
    num_mc_samples=30,    # 30 posterior draws per missing feature
    device="cpu"
)

Running 10 evaluation trials
  Trial 1/10 AUC: 0.9849
  Trial 2/10 AUC: 0.9818
  Trial 3/10 AUC: 0.9835
  Trial 4/10 AUC: 0.9821
  Trial 5/10 AUC: 0.9861
  Trial 6/10 AUC: 0.9843
  Trial 7/10 AUC: 0.9804
  Trial 8/10 AUC: 0.9823
  Trial 9/10 AUC: 0.9820
  Trial 10/10 AUC: 0.9852

Final Model Performance (ROC-AUC): 0.9833 ± 0.0017


In [18]:
torch.save(train_model, r"C:\Users\night\Documents\B.tech CSE(AI)\SEM - 5\22AIE301 Probabilistic Reasoning\project\files\models\model_1_banknote_selfmask.pth")

In [ ]:
# ==========================================
# 1. UPDATED TRAINING FUNCTION
# ==========================================
def train_model(train_loader, input_dim, hidden_dim=64, epochs=100, lr=1e-3, device="cpu"):
    """
    Updated to accept train_loader directly from the experiment loop.
    """
    model = DLGLM(input_dim=input_dim, hidden_dim=hidden_dim).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    for epoch in range(epochs):
        model.train()
        for x_batch, y_batch, mask_batch in train_loader:
            x_batch, y_batch, mask_batch = x_batch.to(device), y_batch.to(device), mask_batch.to(device)
            
            optimizer.zero_grad()
            y_pred, mu, logvar = model(x_batch, mask_batch)
            loss, _, _ = elbo_loss(y_pred, y_batch, mu, logvar, mask_batch)
            loss.backward()
            optimizer.step()
            
    return model

# ==========================================
# 2. EXPERIMENT RUNNER (THE NESTED LOOPS)
# ==========================================
def run_experiments(datasets_info, mechanisms, n_trials=10, num_mc_samples=25, device="cpu"):
    """
    Loops through datasets and missingness mechanisms, trains, evaluates, 
    saves models, and compiles results into a DataFrame.
    """
    results_list = []
    
    # Outer Loop: Iterate through datasets
    for dataset_name, info in datasets_info.items():
        filepath = info['filepath']
        target_col = info['target_col']
        
        print(f"\n{'='*60}")
        print(f"🚀 STARTING DATASET: {dataset_name}")
        print(f"{'='*60}")
        
        # Inner Loop: Iterate through MNAR mechanisms
        for mechanism in mechanisms:
            print(f"\n  ➤ Mechanism: {mechanism}")
            
            # 1. Load Data with specific mechanism
            train_loader, test_loader, input_dim = load_and_prepare_data(
                filepath=filepath, 
                target_col=target_col, 
                mechanism=mechanism, 
                missing_rate=0.5
            )
            
            # 2. Train the Model
            print("    [Training Model...]")
            model = train_model(
                train_loader=train_loader, 
                input_dim=input_dim, 
                device=device
            )
            
            # 3. Evaluate the Model
            print("    [Evaluating Model...]")
            mean_auc, std_auc, _ = evaluate_auc_distribution(
                model=model, 
                test_loader=test_loader, 
                n_trials=n_trials, 
                num_mc_samples=num_mc_samples, 
                device=device
            )
            
            # 4. Save the Model Weights
            # Standard PyTorch practice is to save the state_dict
            model_filename = f"model_{dataset_name}_{mechanism}.pt"
            torch.save(model.state_dict(), model_filename)
            print(f"    💾 Saved weights to: {model_filename}")
            
            # 5. Record Results
            results_list.append({
                'Dataset': dataset_name,
                'Mechanism': mechanism,
                'AUC (Mean)': round(mean_auc, 4),
                'AUC (Std)': round(std_auc, 4),
                'Format (Mean ± Std)': f"{mean_auc:.4f} ± {std_auc:.4f}"
            })

    # Convert results to a pandas DataFrame
    results_df = pd.DataFrame(results_list)
    return results_df

# ==========================================
# 3. EXECUTION BLOCK
# ==========================================
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    # Define your datasets and target columns here
    DATASETS = {
        "BankMarketing": {
            "filepath": "data/bank_marketing.csv",
            "target_col": "y" # e.g., the subscription target
        },
        "MIMIC_subset": {
            "filepath": "data/mimic_clinical.csv",
            "target_col": "mortality_label"
        }
    }
    
    # Define the mechanisms to test
    MECHANISMS = ['self_masking', 'logistic', 'sequential_logistic']
    
    # Run the automated pipeline
    final_results_df = run_experiments(
        datasets_info=DATASETS, 
        mechanisms=MECHANISMS, 
        device=device
    )
    
    # Display the final table
    print("\n\n📊 FINAL EXPERIMENT RESULTS")
    print(final_results_df.to_markdown(index=False))
    
    # Optionally save the DataFrame to a CSV for your report
    final_results_df.to_csv("experiment_results.csv", index=False)